In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [3]:
from tea_data_functions import *

In [4]:
tea_data_path = "../TEA_data/"
econ_disadv = pd.read_csv(tea_data_path+"Economically_Disadvantaged_Report_Statewide_Districts_2024_2025.csv")
grade_enrollment = pd.read_csv(tea_data_path+"Enrollment_Report_Statewide_Districts_Grade_2024_2025.csv")
gender_enrollment = pd.read_csv(tea_data_path+"Enrollment_Report_Statewide_Districts_Gender_2024_2025.csv")
ethnicity_enrollment = pd.read_csv(tea_data_path+"Enrollment_Report_Statewide_Districts_Ethnicity_2024_2025.csv")

In [5]:
grade_enrollment_clean, grade_enrollment_pivot = pivot_enrollment_dataset(grade_enrollment, 'GRADE')
gender_enrollment_clean, gender_enrollment_pivot = pivot_enrollment_dataset(gender_enrollment, 'GENDER')
ethnicity_enrollment_clean, ethnicity_enrollment_pivot = pivot_enrollment_dataset(ethnicity_enrollment, 'ETHNICITY')

In [21]:
econ_disadv_clean = econ_disadv.drop(columns=['Region', ' District Number', ' Charter Status']).rename(columns={' County Name': 'County_Name', ' District Name': 'District_Name', ' Total Count': 'Total_Count'})[['County_Name', 'District_Name', 'Total_Count']].dropna()
econ_disadv_clean['County_Name'] = econ_disadv_clean['County_Name'].str.strip()
econ_disadv_clean['District_Name'] = econ_disadv_clean['District_Name'].str.strip()

In [6]:
common_column_drop = ["CHARTER STATUS", 'REGION', 'YEAR', 'DISTRICT']
econ_disadv_clean = econ_disadv.drop(columns=['Region', ' District Number', ' Charter Status']).dropna()
grade_enrollment_clean = grade_enrollment.drop(columns=common_column_drop).dropna()
gender_enrollment_clean = gender_enrollment.drop(columns=common_column_drop).dropna()
ethnicity_enrollment_clean = ethnicity_enrollment.drop(columns=common_column_drop).dropna()

In [7]:
grade_enrollment_clean[grade_enrollment_clean['ENROLLMENT'].str.contains('<')]['ENROLLMENT'].unique()

<StringArray>
[   '<10',    '<20',    '<30',    '<40',    '<50',    '<80',   '<190',
   '<120',    '<70',   '<130',    '<60',   '<100',   '<110',    '<90',
   '<140',   '<180',   '<320',   '<270',   '<460',   '<150',   '<400',
   '<370',   '<710',   '<200', '<1,260',   '<160',   '<240',   '<830',
   '<570',   '<170',   '<470',   '<350', '<1,010']
Length: 33, dtype: str

In [8]:
gender_enrollment_clean[gender_enrollment_clean['ENROLLMENT'].str.contains('<')]['ENROLLMENT'].unique()

<StringArray>
['<20', '<10', '<50']
Length: 3, dtype: str

In [9]:
ethnicity_enrollment_clean[ethnicity_enrollment_clean['ENROLLMENT'].str.contains('<')]['ENROLLMENT'].unique()

<StringArray>
[ '<10',  '<20',  '<60',  '<50',  '<40',  '<30',  '<70', '<150',  '<90',
 '<120', '<190',  '<80', '<100', '<510', '<340', '<130', '<140', '<730',
 '<170', '<460']
Length: 20, dtype: str

In [58]:
grade_enrollment_pivot.to_csv('../TEA_data/pivoted_grade_enrollment_district_level.csv')

In [52]:
unknown_replace_dict_prep = {'<10': '010', '<20': '020', '<30': '030', '<40': '040','<50': '050', '<60': '060', '<70': '070', '<80': '080', '<90': '89',
                        '<100': '0100', '<110': '0110', '<120': '0120', '<130': '0130', '<140': '0140', '<150': '0150', '<160': '0160', '<170': '0170',
                        '<180': '0180', '<190': '0190', '<200': '0200', '<240': '0240', '<270': '0270', '<320': '0320', '<340': '0340', 
                        '<370': '0370', '<400': '0400', '<460': '0460', '<470': '0470', '<510': '0510', '<570': '0570', '<710': '0710', '<730': '0730',
                        '<830': '0830', '<1,010': '01010', '<1,260': '01260'}

In [59]:
unknown_half_dict = {'<10': '5', '<20': '18', '<30': '15', '<40': '20','<50': '25', '<60': '30', '<70': '35', '<80': '40', '<90': '45',
                        '<100': '50', '<110': '55', '<120': '60', '<130': '65', '<140': '70', '<150': '75', '<160': '80', '<170': '85',
                        '<180': '90', '<190': '95', '<200': '100', '<240': '120', '<270': '135', '<320': '160', '<340': '170', '<350': '175',
                        '<370': '185', '<400': '200', '<460': '230', '<470': '235', '<510': '255', '<570': '285', '<710': '355', '<730': '365',
                        '<830': '415', '<1,010': '505', '<1,260': '630'}

In [60]:
grade_enrollment_prep = grade_enrollment_pivot.replace(unknown_half_dict)
gender_enrollment_prep = gender_enrollment_pivot.replace(unknown_half_dict)
ethnicity_enrollment_prep = ethnicity_enrollment_pivot.replace(unknown_half_dict)

In [61]:
for c in grade_enrollment_prep.drop(columns=['County_Name', 'District_Name']).columns:
    grade_enrollment_prep[c] = grade_enrollment_prep[c].astype(int)

In [62]:
grade_enrollment_prep.head(20)

GRADE,County_Name,District_Name,Early Education,Grade 1,Grade 10,Grade 11,Grade 12,Grade 2,Grade 3,Grade 4,Grade 5,Grade 6,Grade 7,Grade 8,Grade 9,Kindergarten,Pre-kindergarten
0,ANDERSON COUNTY,CAYUGA ISD,5,50,53,51,37,40,43,42,40,41,38,45,49,41,5
1,ANDERSON COUNTY,ELKHART ISD,5,92,87,75,35,81,81,80,85,91,104,88,78,98,81
2,ANDERSON COUNTY,FRANKSTON ISD,5,64,70,67,59,64,47,56,62,64,51,56,63,53,20
3,ANDERSON COUNTY,NECHES ISD,0,21,17,25,27,21,22,36,25,24,26,25,22,31,15
4,ANDERSON COUNTY,PALESTINE ISD,5,210,266,251,228,208,234,206,224,240,225,231,258,100,214
5,ANDERSON COUNTY,SLOCUM ISD,5,25,18,19,23,34,20,23,21,27,24,24,26,21,18
6,ANDERSON COUNTY,WESTWOOD ISD,5,96,108,109,83,101,106,86,111,101,92,108,105,104,40
7,ANDREWS COUNTY,ANDREWS ISD,22,302,319,298,291,322,327,341,297,334,323,313,323,287,110
8,ANGELINA COUNTY,CENTRAL ISD,5,94,113,102,83,121,125,121,115,101,129,106,115,107,35
9,ANGELINA COUNTY,DIBOLL ISD,5,121,114,131,127,97,108,50,106,133,110,113,117,99,106


In [63]:
grade_enrollment_full = pd.merge(grade_enrollment_prep, econ_disadv_clean)

In [64]:
for c in grade_enrollment_full.drop(columns=['County_Name', 'District_Name', 'Total_Count']).columns:
    grade_enrollment_full[c] = round(grade_enrollment_full[c]/grade_enrollment_full['Total_Count'], 3)

In [65]:
grade_enrollment_full.drop(columns=['County_Name', 'District_Name', 'Total_Count']).sum(axis=1)

0       1.006
1       0.978
2       0.988
3       0.998
4       0.973
        ...  
1203    1.043
1204    0.980
1205    1.001
1206    0.971
1207    0.982
Length: 1208, dtype: float64

In [53]:
grade_enrollment_rand = grade_enrollment_pivot.replace(unknown_replace_dict_prep)
gender_enrollment_rand = gender_enrollment_pivot.replace(unknown_replace_dict_prep)
ethnicity_enrollment_rand = ethnicity_enrollment_pivot.replace(unknown_replace_dict_prep)

In [54]:
grade_enrollment_full_rand = pd.merge(grade_enrollment_rand, econ_disadv_clean)

In [56]:
grade_enrollment_full_rand.head()

,County_Name,District_Name,Early Education,Grade 1,Grade 10,Grade 11,Grade 12,Grade 2,Grade 3,Grade 4,Grade 5,Grade 6,Grade 7,Grade 8,Grade 9,Kindergarten,Pre-kindergarten,Total_Count
0,ANDERSON COUNTY,CAYUGA ISD,010,50,53,51,37,40,43,42,40,41,38,45,49,41,010,577.0
1,ANDERSON COUNTY,ELKHART ISD,010,92,87,75,070,81,81,80,85,91,104,88,78,98,81,1187.0
2,ANDERSON COUNTY,FRANKSTON ISD,010,64,70,67,59,64,47,56,62,64,51,56,63,53,040,811.0
3,ANDERSON COUNTY,NECHES ISD,0,21,17,25,27,21,22,36,25,24,26,25,22,31,15,337.0
4,ANDERSON COUNTY,PALESTINE ISD,010,210,266,251,228,208,234,206,224,240,225,231,258,0200,214,3186.0


In [ ]:
for row in grade_enrollment_full_rand.drop(columns=['County_Name', 'District_Name', 'Total_Count']).itertuples(index=False):
    known_numbers = [x for x in list(row) if ]